# Multi-Agent RAG

In [106]:
!pip install openai langchain-openai crewai

## Vectordb setup

In [107]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB with persistence
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(
        allow_reset=True,
        anonymized_telemetry=False
    )
)

## Document Processing Pipeline

In [108]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

class DocumentProcessor:
    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " "]
        )
    
    def process_documents(self, documents):
        chunks = self.text_splitter.split_documents(documents)
        return chunks
        

In [109]:
from langchain_community.document_loaders import PyPDFLoader
document_path = "/home/quang/Downloads/Kinh_te_cong_nghiep.pdf"
loader = PyPDFLoader(document_path)
documents = loader.load()
print(documents)

[Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-07-12T23:23:53+16:23', 'author': 'LeHanh', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-07-12T23:23:53+16:23', 'sourcemodified': "D:20250712232353+16'23'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/home/quang/Downloads/Kinh_te_cong_nghiep.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1'}, page_content='1\nĐẠI HỌC THÁI NGUYÊN\nTRƯỜNG ĐẠI HỌC KỸ THUẬT CÔNG\nNGHIỆP\nCHƯƠNG TRÌNH ĐÀO TẠO TỪ\nXA TRÌNH ĐỘ ĐẠI HỌC\nNGÀNH: KINH TẾ CÔNG NGHIỆP\nCHUYÊN NGÀNH: KẾ TOÁN DOANH\nNGHIỆP CÔNG NGHIỆP'), Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-07-12T23:23:53+16:23', 'author': 'LeHanh', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-07-12T23:23:53+16:23', 'sourcemodified': "D:20250712232353+16'23'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/home/quang/Downloads/Kinh_te_cong_nghiep.pdf', 'total_pages': 53,

In [110]:
raw_texts = [doc.page_content for doc in documents]

chunks = DocumentProcessor().process_documents(documents)

for chunk in chunks:
    chunk.page_content = chunk.page_content.replace("\n", " ")
    chunk.page_content = " ".join(chunk.page_content.split())

In [111]:
print(f"Number of chunks: {chunks[0].page_content}")

Number of chunks: 1 ĐẠI HỌC THÁI NGUYÊN TRƯỜNG ĐẠI HỌC KỸ THUẬT CÔNG NGHIỆP CHƯƠNG TRÌNH ĐÀO TẠO TỪ XA TRÌNH ĐỘ ĐẠI HỌC NGÀNH: KINH TẾ CÔNG NGHIỆP CHUYÊN NGÀNH: KẾ TOÁN DOANH NGHIỆP CÔNG NGHIỆP


In [114]:
import os
from chromadb.config import Settings
import chromadb

# Tạo thư mục nếu chưa tồn tại
persist_dir = "./chroma_db"
os.makedirs(persist_dir, exist_ok=True)

# Khởi tạo client với thư mục này
client = chromadb.Client(Settings(persist_directory=persist_dir))

# Lấy hoặc tạo collection
collection = client.get_or_create_collection(name="MuiltiAgentRAGCollection")

# Thêm dữ liệu
texts = [chunk.page_content for chunk in chunks]
metadatas = [{"source": chunk.metadata.get("source", "unknown")} for chunk in chunks]
embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding", model_kwargs={"device": "cpu"})

collection.add(
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings.embed_documents(texts),
    ids=[str(i) for i in range(len(texts))]
)


## Building Multi-Agent RAG System

### Custom tool websearch

In [ ]:
from crewai.tools import BaseTool
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

class GeminiGoogleSearchTool(BaseTool):
    name:str = "GeminiGoogleSearch"
    description:str = "Search the web using Google's native Gemini Search grounding."

    def _run(self, query: str) -> str:
        # Cấu hình API key
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        
        # Khởi tạo model
        model = genai.GenerativeModel('gemini-pro')
        
        # Tạo prompt bao gồm yêu cầu tìm kiếm
        search_prompt = f"Please search and provide information about: {query}"
        
        # Gọi model để lấy kết quả
        response = model.generate_content(search_prompt)
        
        return response.text

class RAGTool(BaseTool):
    name: str = "RAGTool"
    description: str = "A tool that combines ChromaDB retrieval with web search capabilities"
    
    def __init__(self, retriever, web_search_tool):
        self.retriever = retriever
        self.web_search_tool = web_search_tool
        
    def _run(self, query: str) -> str:
        # First try to get information from ChromaDB
        try:
            docs = self.retriever.get_relevant_documents(query)
            if docs and len(docs) > 0:
                # Combine all relevant documents
                context = "\n\n".join([doc.page_content for doc in docs])
                return f"From local knowledge base:\n{context}"
        except Exception as e:
            print(f"Error retrieving from ChromaDB: {e}")
            
        # If no results from ChromaDB or if there's an error, fallback to web search
        try:
            web_results = self.web_search_tool._run(query)
            return f"From web search:\n{web_results}"
        except Exception as e:
            print(f"Error during web search: {e}")
            return f"Failed to retrieve information from both local database and web search: {str(e)}"

# Initialize tools
gemini_search_tool = GeminiGoogleSearchTool()
rag_tool = RAGTool(retriever=retriever, web_search_tool=gemini_search_tool)

#### Test GoogleSearch

In [116]:
tool = GeminiGoogleSearchTool()
query = "Who won the Euro 2024?"
result = tool.run(query)
print("Query:", query)
print("Result:", result)

Using Tool: GeminiGoogleSearch
Query: Who won the Euro 2024?
Result: Spain won Euro 2024, defeating England 2-1 in the final held in Berlin. This victory marked Spain's record-breaking fourth UEFA European Championship title.

Nico Williams and Mikel Oyarzabal scored the goals for Spain. Cole Palmer scored the equalizer for England. Spain won all seven of their games throughout the tournament. This was England's second consecutive European Championship final loss.
Query: Who won the Euro 2024?
Result: Spain won Euro 2024, defeating England 2-1 in the final held in Berlin. This victory marked Spain's record-breaking fourth UEFA European Championship title.

Nico Williams and Mikel Oyarzabal scored the goals for Spain. Cole Palmer scored the equalizer for England. Spain won all seven of their games throughout the tournament. This was England's second consecutive European Championship final loss.


### Agent Implementation

In [125]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document


In [ ]:
# 1. Tạo embeddings (dùng khi query)
embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding", model_kwargs={"device": "cpu"})

# 2. Kết nối ChromaDB
chroma_db = Chroma(
    persist_directory="./db",  # thư mục bạn đã lưu DB
    embedding_function=embeddings
)

# 3. Tạo retriever
retriever = chroma_db.as_retriever(
    search_kwargs={"k": 5}  # số lượng doc top-k
)


/tmp/ipykernel_5660/889681817.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding", model_kwargs={"device": "cpu"})
/tmp/ipykernel_5660/889681817.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  chroma_db = Chroma(
/tmp/ipykernel_5660/889681817.py:5: LangChainDeprecationWarning: The cla

ValueError: An instance of Chroma already exists for ./chroma_db with different settings

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, WebsiteSearchTool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()

# Kiểm tra OPENAI_API_KEY
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Vui lòng cung cấp OPENAI_API_KEY trong file .env")

# Initialize the language model using OpenAI
llm = ChatOpenAI(
    model="gpt-3.5-turbo",  # Hoặc "gpt-4" nếu bạn cần model mạnh hơn
    temperature=0.7,
    verbose=True
)

# Research Agent
research_agent = Agent(
    role="Research Specialist",
    goal="Retrieve and analyze relevant documents from both local knowledge base and web sources",
    backstory="You are an expert at finding and analyzing relevant information from multiple sources. You first check the local knowledge base for relevant information, and if needed, expand your search to web sources.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[rag_tool]  # Using the combined RAG tool
)

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/alembic/config.py:598: DeprecationWarning: No path_separator found in configuration; falling back to legacy splitting on spaces, commas, and colons for prepend_sys_path.  Consider adding path_separator=os to Alembic config.
  util.warn_deprecated(


### Task Definition and Orchestration

In [128]:
# Research Task
research_task = Task(
    description="Research and answer the following query: {topic}",
    agent=research_agent,
    expected_output="A direct and accurate answer to the query based on available information."
)

### Crew Assembly and Execution

In [129]:
# Create the crew
analysis_crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=True,
    process="sequential"  # or "hierarchical" for complex scenarios
)

# Execute the workflow
def run_analysis(query):
    result = analysis_crew.kickoff(inputs={"topic": query})
    return result

In [130]:
query = "Who won the Euro 2024?"
result = run_analysis(query)
print("Query:", query)
print("Final Result:", result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 84527fd3-f8bf-42c4-84d7-aaa549f9751f                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research and answer the following query: Who won the Euro 2024?                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: Search in a specific website                                                                             │
│  Error: HTTPSConnectionPool(host='www.uefa.com', port=443): Read timed out. (read timeout=30)                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: Search in a specific website                                                                             │
│  Error: HTTPSConnectionPool(host='www.uefa.com', port=443): Read timed out. (read timeout=30)                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 

In [124]:
query = "Mục tiêu (MT) chung cua kinh tế công nghiệp là gì?"
result = run_analysis(query)
print("Query:", query)
print("Final Result:", result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0a2ba2c0-45e4-4c3f-968c-ea806cc9b688                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research and answer the following query: Mục tiêu (MT) chung cua kinh tế công nghiệp là gì?              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Thought: I need to gather information about the general objectives of industrial economics.                    │
│                                                                                                                 │
│  Using Tool: GeminiGoogleSearch                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"query\": \"M\\u1ee5c ti\\u00eau chung c\\u1ee7a kinh t\\u1ebf c\\u00f4ng nghi\\u1ec7p\"}"                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Mục tiêu chung của kinh tế công nghiệp bao gồm nhiều khía cạnh, tập trung vào việc phân tích và giải quyết     │
│  các vấn đề kinh tế trong lĩnh vực công nghiệp để đạt được sự phát triển bền vững và nâng cao hiệu quả hoạt     │
│  động.                                                                                                          │
│                                                                                                                 │
│  Các mục tiêu chính bao gồm:                                                                                    │
│  *   **Thúc đẩy phát triển công nghiệp:** Kinh tế công nghiệp nhằm mục tiêu đạt được sự phát triển công         │
│  nghiệp, cung cấp thông tin liên quan đến tài nguyên thiên nhiên, môi trường công nghiệp để hỗ trợ quá trình    │
│  này. Nó đóng vai trò quan trọng trong việc thúc đẩy tăng trưởng kinh tế tổng thể của một quốc gia.             │
│  *   **Nâng cao hiệu quả và năng suất:** Một trong những mục tiêu trọng tâm là cải thiện hiệu quả công nghiệp   │
│  và năng suất. Điều này bao gồm việc hiểu cách các công ty thiết lập giá cả, công suất, khác biệt hóa sản       │
│  phẩm, đầu tư vào nghiên cứu và phát triển (R&D) và quảng cáo. Nâng cao năng suất chất lượng là động lực phát   │
│  triển bền vững, không chỉ gia tăng sản lượng mà còn gắn liền với hiệu quả sử dụng nguồn lực, khả năng đổi mới  │
│  và mức độ thỏa mãn nhu cầu thị trường.                                                                         │
│  *   **Phân tích và hiểu hành vi của doanh nghiệp và ngành:** Kinh tế công nghiệp nghiên cứu các vấn đề kinh    │
│  tế của các công ty và ngành công nghiệp, cũng như mối quan hệ của chúng với xã hội. Nó tìm cách phát triển     │
│  các giải thích thỏa đáng về cách các lực lượng kinh tế hoạt động trong khu vực công nghiệp, tương tự như mục   │
│  tiêu rộng lớn của lý thuyết kinh tế vi mô. Điều này bao gồm việc đánh giá mức độ cạnh tranh của thị trường,    │
│  điều thường có lợi cho người tiêu dùng.                                                                        │
│  *   **Hỗ trợ hoạch định chính sách:** Kinh tế công nghiệp cung cấp thông tin hữu ích cho các cơ quan quản lý   │
│  của chính phủ để đánh giá sự thành công của chính sách công nghiệp. Các chính sách công nghiệp trọng điểm      │
│  thường hướng tới ba mục tiêu: nâng cấp công nghệ, chuyển dịch cơ cấu và cạnh tranh quốc tế.                    │
│  *   **Tạo việc làm và nâng cao chất lượng cuộc sống:** Ngành công nghiệp tạo ra nhiều cơ hội việc làm, từ      │
│  công nhân sản xuất đến kỹ sư, quản lý và góp phần nâng cao chất lượng cuộc sống.                               │
│  ...                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Mục tiêu chung của kinh tế công nghiệp bao gồm nhiều khía cạnh, tập trung vào việc phân tích và giải quyết     │
│  các vấn đề kinh tế trong lĩnh vực công nghiệp để đạt được sự phát triển bền vững và nâng cao hiệu quả hoạt     │
│  động.                                                                                                          │
│                                                                                                                 │
│  Các mục tiêu chính bao gồm:                                                                                    │
│  *   **Thúc đẩy phát triển công nghiệp:** Kinh tế công nghiệp nhằm mục tiêu đạt được sự phát triển công         │
│  nghiệp, cung cấp thông tin liên quan đến tài nguyên thiên nhiên, môi trường công nghiệp để hỗ trợ quá trình    │
│  này. Nó đóng vai trò quan trọng trong việc thúc đẩy tăng trưởng kinh tế tổng thể của một quốc gia.             │
│  *   **Nâng cao hiệu quả và năng suất:** Một trong những mục tiêu trọng tâm là cải thiện hiệu quả công nghiệp   │
│  và năng suất. Điều này bao gồm việc hiểu cách các công ty thiết lập giá cả, công suất, khác biệt hóa sản       │
│  phẩm, đầu tư vào nghiên cứu và phát triển (R&D) và quảng cáo. Nâng cao năng suất chất lượng là động lực phát   │
│  triển bền vững, không chỉ gia tăng sản lượng mà còn gắn liền với hiệu quả sử dụng nguồn lực, khả năng đổi mới  │
│  và mức độ thỏa mãn nhu cầu thị trường.                                                                         │
│  *   **Phân tích và hiểu hành vi của doanh nghiệp và ngành:** Kinh tế công nghiệp nghiên cứu các vấn đề kinh    │
│  tế của các công ty và ngành công nghiệp, cũng như mối quan hệ của chúng với xã hội. Nó tìm cách phát triển     │
│  các giải thích thỏa đáng về cách các lực lượng kinh tế hoạt động trong khu vực công nghiệp, tương tự như mục   │
│  tiêu rộng lớn của lý thuyết kinh tế vi mô. Điều này bao gồm việc đánh giá mức độ cạnh tranh của thị trường,    │
│  điều thường có lợi cho người tiêu dùng.                                                                        │
│  *   **Hỗ trợ hoạch định chính sách:** Kinh tế công nghiệp cung cấp thông tin hữu ích cho các cơ quan quản lý   │
│  của chính phủ để đánh giá sự thành công của chính sách công nghiệp. Các chính sách công nghiệp trọng điểm      │
│  thường hướng tới ba mục tiêu: nâng cấp công nghệ, chuyển dịch cơ cấu và cạnh tranh quốc tế.                    │
│  *   **Tạo việc làm và nâng cao chất lượng cuộc sống:** Ngành công nghiệp tạo ra nhiều cơ hội việc làm, từ      │
│  công nhân sản xuất đến kỹ sư, quản lý và góp phần nâng cao chất lượng cuộc sống.                               │
│  *   **Đổi mới công nghệ và nâng cao năng lực cạnh tranh quốc gia:** Kinh tế công nghiệp khuyến khích đổi mới   │
│  công nghệ và ứng dụng các công nghệ mới nhất. Một nền công nghiệp hiện đại và năng động giúp tăng cường năng   │
│  lực cạnh tranh của quốc gia trên thị trường quốc tế.                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 421cbc69-b6e5-44cb-9e63-0cef5edc01e4                                                                     │
│  Agent: Research Specialist                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0a2ba2c0-45e4-4c3f-968c-ea806cc9b688                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Mục tiêu chung của kinh tế công nghiệp bao gồm nhiều khía cạnh, tập trung vào việc phân tích và  │
│  giải quyết các vấn đề kinh tế trong lĩnh vực công nghiệp để đạt được sự phát triển bền vững và nâng cao hiệu   │
│  quả hoạt động.                                                                                                 │
│                                                                                                                 │
│  Các mục tiêu chính bao gồm:                                                                                    │
│  *   **Thúc đẩy phát triển công nghiệp:** Kinh tế công nghiệp nhằm mục tiêu đạt được sự phát triển công         │
│  nghiệp, cung cấp thông tin liên quan đến tài nguyên thiên nhiên, môi trường công nghiệp để hỗ trợ quá trình    │
│  này. Nó đóng vai trò quan trọng trong việc thúc đẩy tăng trưởng kinh tế tổng thể của một quốc gia.             │
│  *   **Nâng cao hiệu quả và năng suất:** Một trong những mục tiêu trọng tâm là cải thiện hiệu quả công nghiệp   │
│  và năng suất. Điều này bao gồm việc hiểu cách các công ty thiết lập giá cả, công suất, khác biệt hóa sản       │
│  phẩm, đầu tư vào nghiên cứu và phát triển (R&D) và quảng cáo. Nâng cao năng suất chất lượng là động lực phát   │
│  triển bền vững, không chỉ gia tăng sản lượng mà còn gắn liền với hiệu quả sử dụng nguồn lực, khả năng đổi mới  │
│  và mức độ thỏa mãn nhu cầu thị trường.                                                                         │
│  *   **Phân tích và hiểu hành vi của doanh nghiệp và ngành:** Kinh tế công nghiệp nghiên cứu các vấn đề kinh    │
│  tế của các công ty và ngành công nghiệp, cũng như mối quan hệ của chúng với xã hội. Nó tìm cách phát triển     │
│  các giải thích thỏa đáng về cách các lực lượng kinh tế hoạt động trong khu vực công nghiệp, tương tự như mục   │
│  tiêu rộng lớn của lý thuyết kinh tế vi mô. Điều này bao gồm việc đánh giá mức độ cạnh tranh của thị trường,    │
│  điều thường có lợi cho người tiêu dùng.                                                                        │
│  *   **Hỗ trợ hoạch định chính sách:** Kinh tế công nghiệp cung cấp thông tin hữu ích cho các cơ quan quản lý   │
│  của chính phủ để đánh giá sự thành công của chính sách công nghiệp. Các chính sách công nghiệp trọng điểm      │
│  thường hướng tới ba mục tiêu: nâng cấp công nghệ, chuyển dịch cơ cấu và cạnh tranh quốc tế.                    │
│  *   **Tạo việc làm và nâng cao chất lượng cuộc sống:** Ngành công nghiệp tạo ra nhiều cơ hội việc làm, từ      │
│  công nhân sản xuất đến kỹ sư, quản lý và góp phần nâng cao chất lượng cuộc sống.                               │
│  *   **Đổi mới công nghệ và nâng cao năng lực cạnh tranh quốc gia:** Kinh tế công nghiệp khuyến khích đổi mới   │
│  công nghệ và ứng dụng các công nghệ mới nhất. Một nền công nghiệp hiện đại và năng động giúp tăng cường năng   │
│  lực cạnh tranh của quốc gia trên thị trường quốc tế.                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

Query: Mục tiêu (MT) chung cua kinh tế công nghiệp là gì?
Final Result: Mục tiêu chung của kinh tế công nghiệp bao gồm nhiều khía cạnh, tập trung vào việc phân tích và giải quyết các vấn đề kinh tế trong lĩnh vực công nghiệp để đạt được sự phát triển bền vững và nâng cao hiệu quả hoạt động.

Các mục tiêu chính bao gồm:
*   **Thúc đẩy phát triển công nghiệp:** Kinh tế công nghiệp nhằm mục tiêu đạt được sự phát triển công nghiệp, cung cấp thông tin liên quan đến tài nguyên thiên nhiên, môi trường công nghiệp để hỗ trợ quá trình này. Nó đóng vai trò quan trọng trong việc thúc đẩy tăng trưởng kinh tế tổng thể của một quốc gia.
*   **Nâng cao hiệu quả và năng suất:** Một trong những mục tiêu trọng tâm là cải thiện hiệu quả công nghiệp và năng suất. Điều này bao gồm việc hiểu cách các công ty thiết lập giá cả, công suất, khác biệt hóa sản phẩm, đầu tư vào nghiên cứu và phát triển (R&D) và quảng cáo. Nâng cao năng suất chất lượng là động lực phát triển bền vững, không chỉ gia tăng sản lượng m

## Advanced Features and Production Considerations

### Memory and Context Management


In [ ]:
from crewai.memory import ShortTermMemory, LongTermMemory

# Configure memory systems
short_term_memory = ShortTermMemory(
    provider="chroma",
    config={"collection_name": "agent_short_term_memory"}
)

long_term_memory = LongTermMemory(
    provider="chroma",
    config={"collection_name": "agent_long_term_memory"}
)

# Apply to agents
research_agent.memory = short_term_memory
analysis_agent.memory = long_term_memory

TypeError: ShortTermMemory.__init__() got an unexpected keyword argument 'provider'

### Error Handling and Resilience


In [ ]:
import time
class ResilientCrew:
    def __init__(self, crew, max_retries=3):
        self.crew = crew
        self.max_retries = max_retries

    def execute_with_retry(self, inputs):
        for attempt in range(self.max_retries):
            try:
                return self.crew.kickoff(inputs=inputs)
            except Exception as e:
                if attempt == self.max_retries - 1:
                    raise e
                print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
                time.sleep(2 ** attempt)  # Exponential backoff


### Monitoring and Observability


In [ ]:
import logging
from datetime import datetime

class AgentMonitor:
    def __init__(self):
        self.logger = logging.getLogger("multi_agent_rag")
        self.metrics = {}

    def log_agent_performance(self, agent_name, task_duration, success):
        self.logger.info(f"Agent: {agent_name}, Duration: {task_duration}s, Success: {success}")

        if agent_name not in self.metrics:
            self.metrics[agent_name] = {"total_tasks": 0, "successful_tasks": 0, "avg_duration": 0}

        self.metrics[agent_name]["total_tasks"] += 1
        if success:
            self.metrics[agent_name]["successful_tasks"] += 1

        # Update average duration
        current_avg = self.metrics[agent_name]["avg_duration"]
        total_tasks = self.metrics[agent_name]["total_tasks"]
        self.metrics[agent_name]["avg_duration"] = (current_avg * (total_tasks - 1) + task_duration) / total_tasks

### Horizontal Scaling


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import queue

class AgentPool:
    def __init__(self, agent_factory, pool_size=5):
        self.agents = [agent_factory() for _ in range(pool_size)]
        self.available_agents = queue.Queue()
        for agent in self.agents:
            self.available_agents.put(agent)

    def execute_task(self, task_data):
        agent = self.available_agents.get()
        try:
            result = agent.execute(task_data)
            return result
        finally:
            self.available_agents.put(agent)

### Performance Optimization


In [ ]:
from functools import lru_cache
import asyncio

class OptimizedRAGSystem:
    def __init__(self):
        self.vector_cache = {}
        self.response_cache = {}

    @lru_cache(maxsize=1000)
    def cached_retrieval(self, query_hash):
        # Implement cached document retrieval
        pass

    async def batch_process(self, queries):
        # Process multiple queries in parallel
        tasks = [self.process_query(query) for query in queries]
        return await asyncio.gather(*tasks)
